# 01 — Data Ingestion

Primary source: **Supabase** (BLE sessions from the PWA app).  
Fallback: local CSV files in `data/raw/` (original pipeline).

Outputs:
- `data/processed/master_long.parquet` — 1Hz time series (for notebooks 02 and 03)
- `data/processed/hrv_master.parquet` — one row per session with real HRV metrics
  (only when data comes from Supabase with real RR intervals)

In [1]:
import sys, re, logging
from pathlib import Path

import pandas as pd
import numpy as np

REPO_ROOT = Path("..").resolve()
sys.path.insert(0, str(REPO_ROOT))

from src.polar_parser import parse_polar_csv
from src.hrv_metrics  import compute_hrv

RAW_DIR       = REPO_ROOT / "data" / "raw"
PROCESSED_DIR = REPO_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
logger = logging.getLogger("01_ingestion")

print(f"REPO_ROOT     : {REPO_ROOT}")
print(f"PROCESSED_DIR : {PROCESSED_DIR}")

REPO_ROOT     : /home/bruno1008/Neroes/neroes_polar_pipeline
PROCESSED_DIR : /home/bruno1008/Neroes/neroes_polar_pipeline/data/processed


In [2]:
# ── Primary source: Supabase ────────────────────────────────────────────────
# Connects with service_role key from config/secrets.yaml.
# Falls back to CSV files automatically if Supabase is unreachable or empty.

supabase_ok       = False
master_long_raw   = pd.DataFrame()
supabase_sessions = pd.DataFrame()

try:
    from src.supabase_loader import build_master_dataframe, get_sessions, get_rr_intervals
    master_long_raw   = build_master_dataframe()
    supabase_sessions = get_sessions()
    if not master_long_raw.empty:
        supabase_ok = True
        n_p = master_long_raw["User_ID"].nunique()
        n_d = master_long_raw["Date"].nunique()
        print(f"Supabase: {len(master_long_raw):,} rows from {n_p} participant(s), {n_d} date(s).")
    else:
        print("Supabase: connected but no sessions found → falling back to CSV files.")
except Exception as exc:
    print("Supabase unavailable → falling back to CSV files.")
    print(f"  ({type(exc).__name__}: {exc})")

INFO | HTTP Request: GET https://guwpubswzbdvpiasnxjh.supabase.co/rest/v1/sessions?select=%2A%2Cparticipants%28code%2Cname%2Cbirthdate%2Cgender%2Cheight_cm%2Cweight_kg%29&order=session_date.asc%2Csession_time.asc "HTTP/2 200 OK"


INFO | HTTP Request: GET https://guwpubswzbdvpiasnxjh.supabase.co/rest/v1/rr_intervals?select=seq%2Crr_ms&session_id=eq.d9e152be-f4ca-4e76-baf5-badb731e67f3&order=seq.asc "HTTP/2 200 OK"


INFO | polar01    | 2026-06-18 | session d9e152be | 28 RR → 21 rows @1Hz


INFO | HTTP Request: GET https://guwpubswzbdvpiasnxjh.supabase.co/rest/v1/sessions?select=%2A%2Cparticipants%28code%2Cname%2Cbirthdate%2Cgender%2Cheight_cm%2Cweight_kg%29&order=session_date.asc%2Csession_time.asc "HTTP/2 200 OK"


Supabase: 21 rows from 1 participant(s), 1 date(s).


In [3]:
# ── Fallback: local CSV files ────────────────────────────────────────────────
if not supabase_ok:
    _DATE_RE = re.compile(r"(\d{4}-\d{2}-\d{2})")

    def _extract_date(path):
        m = _DATE_RE.search(path.stem)
        try: return pd.to_datetime(m.group(1)).date() if m else None
        except Exception: return None

    # Exclude rr/ subdirectories (those are API-downloaded interval files)
    csv_files = sorted([*RAW_DIR.rglob("*.csv"), *RAW_DIR.rglob("*.CSV")])
    csv_files = [f for f in csv_files
                 if not f.name.startswith(".") and "rr" not in [p.lower() for p in f.parts]]

    print(f"Found {len(csv_files)} CSV file(s)")
    all_frames, errors = [], []

    for csv_path in csv_files:
        user_id = csv_path.parent.name
        date    = _extract_date(csv_path)
        result  = parse_polar_csv(csv_path)
        ts, meta = result["timeseries"], result["metadata"]

        if ts.empty:
            logger.warning("Skipping %s — empty time-series", csv_path.name)
            errors.append(csv_path.name)
            continue

        ts = ts.copy()
        ts["User_ID"]     = user_id
        ts["Date"]        = pd.to_datetime(date) if date else pd.NaT
        ts["Day_of_Week"] = pd.to_datetime(date).day_name() if date else None
        ts["hrv_source"]  = "bpm_estimated"
        ts["session_id"]  = None
        for key, val in meta.items():
            if key not in ts.columns:
                ts[key] = val
        all_frames.append(ts)
        logger.info("CSV: %-40s | rows=%d | user=%s", csv_path.name, len(ts), user_id)

    if all_frames:
        master_long_raw = pd.concat(all_frames, ignore_index=True)
    if errors:
        print(f"WARNING: {len(errors)} file(s) skipped: {errors}")

print(f"master_long shape: {master_long_raw.shape}")

master_long shape: (21, 11)


In [4]:
# ── HRV metrics from raw RR intervals (Supabase / BLE path only) ────────────
# When data comes from Supabase, each session has real beat-to-beat RR intervals.
# compute_hrv() applies artifact filtering and returns scientifically valid metrics.
# When data comes from CSV (1Hz HR), this block is skipped; feature_engineering.py
# will compute an approximation (hrv_source = "bpm_estimated") in notebook 02.

hrv_rows = []

if supabase_ok and not supabase_sessions.empty:
    for _, sess in supabase_sessions.iterrows():
        rr  = get_rr_intervals(str(sess["id"]))
        hrv = compute_hrv(rr)
        hrv_rows.append({
            "session_id"  : str(sess["id"]),
            "User_ID"     : sess.get("participant_code", ""),
            "Date"        : pd.to_datetime(sess["session_date"]),
            "session_time": str(sess.get("session_time", "")),
            "n_rr_raw"    : len(rr),
            **hrv,
        })

    print(f"HRV computed for {len(hrv_rows)} session(s):")
    if hrv_rows:
        display(pd.DataFrame(hrv_rows)[[
            "User_ID", "Date", "n_rr", "hr_resting_mean",
            "rmssd", "lnrmssd", "pnn50", "quality_flag"
        ]])
else:
    print("CSV path — HRV will be estimated in notebook 02 from 1Hz HR.")
    print("hrv_master.parquet will be empty until BLE sessions arrive.")

INFO | HTTP Request: GET https://guwpubswzbdvpiasnxjh.supabase.co/rest/v1/rr_intervals?select=seq%2Crr_ms&session_id=eq.d9e152be-f4ca-4e76-baf5-badb731e67f3&order=seq.asc "HTTP/2 200 OK"


HRV computed for 1 session(s):


,User_ID,Date,n_rr,hr_resting_mean,rmssd,lnrmssd,pnn50,quality_flag
0,polar01,2026-06-18,28,82.86,26.379,3.2726,14.81,good


In [5]:
# ── Save master_long.parquet ─────────────────────────────────────────────────
if master_long_raw.empty:
    print("⚠️  No data loaded.")
    print("  → Add CSV files to data/raw/<participant>/ OR record sessions via the app.")
else:
    out_long = PROCESSED_DIR / "master_long.parquet"
    master_long_raw.to_parquet(out_long, index=False, engine="pyarrow")
    print(f"Saved → {out_long.relative_to(REPO_ROOT)}  shape={master_long_raw.shape}")

# ── Save hrv_master.parquet ───────────────────────────────────────────────────
hrv_master = pd.DataFrame(hrv_rows) if hrv_rows else pd.DataFrame()
out_hrv    = PROCESSED_DIR / "hrv_master.parquet"
hrv_master.to_parquet(out_hrv, index=False, engine="pyarrow")
if hrv_master.empty:
    print(f"Saved → {out_hrv.relative_to(REPO_ROOT)}  (empty — no BLE sessions yet)")
else:
    print(f"Saved → {out_hrv.relative_to(REPO_ROOT)}  shape={hrv_master.shape}")

Saved → data/processed/master_long.parquet  shape=(21, 11)
Saved → data/processed/hrv_master.parquet  shape=(1, 16)


In [6]:
# ── Summary ──────────────────────────────────────────────────────────────────
if not master_long_raw.empty:
    src  = "Supabase (BLE RR intervals)" if supabase_ok else "Local CSV files (1Hz HR)"
    n_p  = master_long_raw["User_ID"].nunique()
    n_s  = master_long_raw.groupby("User_ID")["Date"].nunique().sum()
    n_rr = len(hrv_rows)

    print(f"Data source  : {src}")
    print(f"Participants : {n_p}")
    print(f"Sessions     : {n_s}")
    print(f"Records      : {len(master_long_raw):,}  (@ 1 Hz interpolated)")
    if hrv_master is not None and not hrv_master.empty:
        print(f"HRV sessions : {n_rr} (real RR intervals)")
    print()
    display(master_long_raw.groupby("User_ID")["Date"].nunique().rename("sessions").to_frame())

Data source  : Supabase (BLE RR intervals)
Participants : 1
Sessions     : 1
Records      : 21  (@ 1 Hz interpolated)
HRV sessions : 1 (real RR intervals)



,sessions
User_ID,
polar01,1
